In [40]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os
import pdb

In [44]:

def plot_cluster_violin_grid(csv_file, K, row_spacing=0.5, add_noise = False):
    # Load CSV
    df = pd.read_csv(csv_file)

    # Check required columns
    required_cols = {'cluster_id', 'event_template_id', 'event_matching_id', 'path_id', 'pid', 'mean_each_pid'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"Missing columns. Required: {required_cols}")
    unique_clusters = df['cluster_id'].unique() if not add_noise else [2]
    for cluster_id in unique_clusters:
        cluster_df = df[df['cluster_id'] == cluster_id]

        template_ids = sorted(cluster_df['event_template_id'].unique())
        matching_ids = sorted(cluster_df['event_matching_id'].unique())
        path_ids = ['sameEv-otherSchema', 'otherEv-otherSchema', 'otherEv-sameSchema', 'sameEv-sameSchema']

        fig = plt.figure(figsize=(18, 18), dpi = 500)
        fig.suptitle(f'Cluster ID: {cluster_id}', fontsize=20)

        # Define GridSpec with extra vertical spacing
        gs = gridspec.GridSpec(nrows=3, ncols=3, figure=fig, hspace=row_spacing)

        palette = ["royalblue", "tomato", "yellow", "mediumorchid"]
        path_color_dict = dict(zip(path_ids, palette))

        for i, match_id in enumerate(matching_ids):
            for j, temp_id in enumerate(template_ids):
                ax = fig.add_subplot(gs[i, j])

                subset = cluster_df[
                    (cluster_df['event_matching_id'] == match_id) &
                    (cluster_df['event_template_id'] == temp_id)
                ]

                if add_noise:
                    subset_mean_each_pid_np = subset["mean_each_pid"].to_numpy()
                    noise = np.random.normal(0,.05, size = subset_mean_each_pid_np.shape)
                    subset["mean_each_pid"] = (subset_mean_each_pid_np + noise).tolist()
                    print(np.all(subset_mean_each_pid_np == (subset_mean_each_pid_np + noise)))

                if not subset.empty:
                    sns.violinplot(
                        data=subset,
                        x='path_id',
                        y='mean_each_pid',
                        ax=ax,
                        inner='box',
                        palette=path_color_dict,
                        linewidth=1,
                        order=path_ids
                    )

                    ax.axhline(0, color='black', linestyle='-', linewidth=2)
                    ax.set_xticks([])
                    ax.set_xlabel('')
                    ax.set_ylabel('neural similarity' if j == 0 else '')
                    ax.set_title(f'stage-{temp_id}-template / apply to ritual {match_id}', fontsize=10)
        
        # Save and show
        plt.savefig(f'K{K}/collapseTR_K{K}_cluster{cluster_id}_3by3_addNoise{add_noise}.png')
        plt.show()
        if add_noise:
            break


In [45]:

def plot_cluster_violin_with_mean(csv_file, output_dir='cluster_plots'):
    # Load data
    df = pd.read_csv(csv_file)

    # Get unique cluster IDs
    cluster_ids = df['cluster_id'].unique()

    for cluster_id in cluster_ids:
        cluster_data = df[df['cluster_id'] == cluster_id]

        # Split data into matching and non-matching
        matching = cluster_data[cluster_data['event_template_id'] == cluster_data['event_matching_id']]
        non_matching = cluster_data[cluster_data['event_template_id'] != cluster_data['event_matching_id']]

        # Get consistent path_id order and colors
        path_ids = ['sameEv-otherSchema','otherEv-otherSchema','otherEv-sameSchema','sameEv-sameSchema']
        palette = ["royalblue", "tomato", "yellow", "mediumorchid"]#sns.color_palette('Set2', len(path_ids))
        path_color_map = dict(zip(path_ids, palette))

        # Create subplots
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
        fig.suptitle(f'Cluster ID: {cluster_id}', fontsize=16)

        def plot_violin(ax, data, title):
            sns.violinplot(
                x='path_id', y='mean_each_pid', data=data,
                ax=ax, order=path_ids, palette=path_color_map, inner='box', alpha = 0.5
            )
            ax.set_title(title)
            ax.set_xlabel('')
            ax.set_ylabel('neural similarity')
            ax.set_xticks([])  # Remove x-ticks
            ax.axhline(0, color='black', linestyle='-', linewidth=2)


            # Add colored horizontal lines at each mean
            # for i, path_id in enumerate(path_ids):
            #     mean_val = data[data['path_id'] == path_id]['mean_each_pid'].mean()
            #     ax.axhline(y=mean_val, color=path_color_map[path_id], linestyle='-', linewidth=2)

        # Plot subplots
        plot_violin(axes[0], matching, 'within-stage')
        plot_violin(axes[1], non_matching, 'across-stage')

        # Add a legend at the bottom
        handles = [
            plt.Line2D([0], [0], color=path_color_map[pid], lw=4)
            for pid in path_ids
        ]


        
        labels = [f'{pid}' for pid in path_ids]
        #fig.legend(handles, labels, loc='lower center', ncol=len(path_ids), fontsize=12, title_fontsize=13)
        plt.savefig(f'K{K}/collapseTR_K{K}_cluster{cluster_id}_within+across.png')

        plt.tight_layout(rect=[0, 0.05, 1, 0.95])
        plt.show()


# Example usage:
# plot_cluster_violin_with_mean('your_data.csv')


In [46]:
for K in range(6,7):
    path_to_visuals = f"/scratch/gpfs/rk1593/clustering_output/kmeans_fingerprints_tval_collapseTR/kmeans_{K}clusters_collapseTR.csv"
    # Example usage
    plot_cluster_violin_grid(path_to_visuals, K, add_noise = True)
 #   plot_cluster_violin_with_mean(path_to_visuals, K)

<ipython-input-44-3ca8744b3c29>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset["mean_each_pid"] = (subset_mean_each_pid_np + noise).tolist()
<ipython-input-44-3ca8744b3c29>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset["mean_each_pid"] = (subset_mean_each_pid_np + noise).tolist()


False
False


<ipython-input-44-3ca8744b3c29>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset["mean_each_pid"] = (subset_mean_each_pid_np + noise).tolist()
<ipython-input-44-3ca8744b3c29>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset["mean_each_pid"] = (subset_mean_each_pid_np + noise).tolist()
<ipython-input-44-3ca8744b3c29>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the do

False
False
False
False
False


<ipython-input-44-3ca8744b3c29>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset["mean_each_pid"] = (subset_mean_each_pid_np + noise).tolist()
<ipython-input-44-3ca8744b3c29>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset["mean_each_pid"] = (subset_mean_each_pid_np + noise).tolist()


False
False
